# Jebrex AI — Phase F Cloud GPU Production Training Notebook (`125M` -> `350M`)

Staged, gate-enforced cloud training notebook for Google Colab, Kaggle, or Cloud GPU hosts:
1. **Stage 1 (`configs/training_125m.json`):** `jebrex-125m` (`124,275,456` parameters, RoPE, RMSNorm, SwiGLU, GQA + KV-Cache, Gradient Checkpointing, BF16).
2. **Stage 2 (`configs/training_350m.json`):** `jebrex-350m` (`351,323,136` parameters) — enabled ONLY after Stage 1 passes held-out evaluation and >=24 GiB VRAM + >=10B tokens are provisioned.

**Mandatory Scientific & Cost Safety Gates:**
- Production training is **HARD-BLOCKED** if `overall_production_readiness_gate != 'DATA_READY'` or CUDA VRAM is insufficient.
- Checkpoints are NEVER committed to GitHub; they sync as **SHA-256 Portable Bundles** to external cloud storage (Google Drive / GCS / Cloudflare R2).
- Session interruptions automatically resume exact `AdamW` + `Scheduler` + `RNG` + `GradScaler` state.

In [ ]:
# 1. Detect GPU Hardware & Mount Persistent External Storage (Google Drive / Cloud Disk)
import os, sys, json, torch
print('PyTorch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {props.name} | VRAM: {props.total_memory / (1024**3):.2f} GiB | BF16: {torch.cuda.is_bf16_supported()}')
else:
    print('WARNING: No CUDA GPU detected. Production 125M/350M gate will BLOCK training until GPU runtime is selected.')

try:
    from google.colab import drive
    drive.mount('/content/drive')
    EXTERNAL_STORAGE_ROOT = '/content/drive/MyDrive/JebrexAI/external_bundles'
except ImportError:
    EXTERNAL_STORAGE_ROOT = './external_cloud_bundles'
os.makedirs(EXTERNAL_STORAGE_ROOT, exist_ok=True)
print('External SHA-256 bundle root:', EXTERNAL_STORAGE_ROOT)

In [ ]:
# 2. Clone / Update Repository (Code & Manifests Only — Weights Stay External)
!git clone --depth 1 -b comprehensive_project_audit https://github.com/ypy999/NourAI.git jebrex_ai || (cd jebrex_ai && git pull)
%cd jebrex_ai
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

In [ ]:
# 3. Mandatory Production Training Preflight Gate (Stage 1: 125M first, Stage 2: 350M second)
from pathlib import Path
from training.production_launcher import evaluate_production_training_gate

STAGE_CONFIG = Path('configs/training_125m.json')  # Switch to configs/training_350m.json ONLY after Stage 1 passes
DATASET_MANIFEST = Path('data/manifests/jebrex_v1_clean.v1.0.0.manifest.json')  # Point to your multi-billion licensed manifest
BPE_TOKENIZER_PATH = Path('tokenizer/artifacts/jebrex_bpe_v1.json')  # Point to your 32,768-vocab BPE artifact

gate_report = evaluate_production_training_gate(
    config_path=STAGE_CONFIG,
    dataset_manifest_path=DATASET_MANIFEST,
    bpe_path=BPE_TOKENIZER_PATH,
    report_out=None,
)
print('Gate Status:', gate_report['gate_status'])
print('Blockers:', json.dumps(gate_report['blockers'], ensure_ascii=False, indent=2))
if not gate_report['training_allowed']:
    raise RuntimeError('Production training HALTED: resolve all gate blockers above before running Stage 1/2 training.')

In [ ]:
# 4. Execute Resumable Production Training + External SHA-256 Bundle Sync
from model.jebrex_transformer import JebrexConfig, JebrexTransformer
from tokenizer.bpe_tokenizer import MultilingualBPETokenizer
from datasets.efficient_loader import pack_jsonl_to_uint16_bin, build_memmap_dataloader
from training.cloud_trainer import CloudTrainConfig, CloudResumableTrainer
from training.production_launcher import ExternalCheckpointStorage

spec = json.loads(STAGE_CONFIG.read_text(encoding='utf-8'))
ds_manifest = json.loads(DATASET_MANIFEST.read_text(encoding='utf-8'))
tok = MultilingualBPETokenizer.load(BPE_TOKENIZER_PATH)

run_dir = Path('./local_gpu_run') / spec['preset']
bin_dir = run_dir / 'packed_bin'
pack_jsonl_to_uint16_bin(Path(ds_manifest['splits']['train']['path']), bin_dir / 'train.bin', tok)
pack_jsonl_to_uint16_bin(Path(ds_manifest['splits']['validation']['path']), bin_dir / 'val.bin', tok)
pack_jsonl_to_uint16_bin(Path(ds_manifest['splits']['test']['path']), bin_dir / 'test.bin', tok)

model_cfg = JebrexConfig.from_dict(spec['model'])
t_spec = spec['training']
train_cfg = CloudTrainConfig(
    preset=spec['preset'],
    batch_size=t_spec['batch_size'],
    grad_accum_steps=t_spec['grad_accum_steps'],
    block_size=t_spec['block_size'],
    max_steps=t_spec['max_steps'],
    warmup_steps=t_spec['warmup_steps'],
    learning_rate=t_spec['learning_rate'],
    min_lr_ratio=t_spec['min_lr_ratio'],
    weight_decay=t_spec['weight_decay'],
    grad_clip=t_spec['grad_clip'],
    eval_interval=t_spec['eval_interval'],
    save_interval=t_spec['save_interval'],
    keep_last_checkpoints=t_spec['keep_last_checkpoints'],
    gradient_checkpointing=True,
    mixed_precision=t_spec['mixed_precision'],
    remote_uri=spec['external_storage']['remote_uri_template'],
)

train_loader = build_memmap_dataloader(bin_dir / 'train.bin', train_cfg.block_size, train_cfg.batch_size, shuffle=True)
val_loader = build_memmap_dataloader(bin_dir / 'val.bin', train_cfg.block_size, train_cfg.batch_size, shuffle=False)
test_loader = build_memmap_dataloader(bin_dir / 'test.bin', train_cfg.block_size, train_cfg.batch_size, shuffle=False)

model = JebrexTransformer(model_cfg)
trainer = CloudResumableTrainer(model, train_cfg, run_dir)
ext_store = ExternalCheckpointStorage(EXTERNAL_STORAGE_ROOT)
bundle_name = f"{spec['preset']}_latest_bundle"

# Auto-recover from external SHA-256 bundle if a prior cloud session was preempted
if (Path(EXTERNAL_STORAGE_ROOT) / bundle_name / 'manifest.json').is_file():
    pulled = ext_store.pull_portable_bundle(bundle_name, run_dir / 'recovered_bundle')
    print('Recovered from external bundle:', trainer.import_portable_bundle(Path(pulled['local_bundle_dir'])))

session_summary = trainer.train(train_loader, val_loader)
print('Training summary:', json.dumps(session_summary, indent=2))

# Export and SHA-256 verify to external cloud storage
local_bundle = run_dir / 'export_bundle'
trainer.export_portable_bundle(local_bundle)
sync_result = ext_store.push_portable_bundle(local_bundle, bundle_name)
print('External SHA-256 sync:', json.dumps(sync_result, indent=2))
print('Post-training held-out test metrics:', trainer.evaluate(test_loader))